# EarthScope electric-field calibration workflow

This notebook is a working prototype for adding electric-field handling to the EarthScope tutorial. The goal is to show the end-to-end pattern used in the existing magnetic-data notebook, but extended to `Q` channels.

The example downloads a short EarthScope run from station `CAS04`, calibrates the electric and magnetic channels, and plots the resulting time series.


## 1. Discover the electric channels

EarthScope uses the SEED instrument code `Q` for electric-field channels, while `F` is used for magnetic field. For a first pass, a query using `?QE`, `?QN`, `?QZ`, or station-specific labels such as `LQE`, `LQN`, `LQ2`, and `LQ3` is the easiest way to find usable data.

This example uses a known-good station, CAS04, where the archived metadata already includes the electric dipole-length response and the calibration can be applied directly.


In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from mth5.clients import FDSN
from mth5.mth5 import MTH5


In [ ]:
fdsn = FDSN(mth5_version='0.2.0')
fdsn.client = 'EARTHSCOPE'

network = '8P'
station = 'CAS04'
start = '2020-06-02T19:00:00'
end = '2020-06-02T20:00:00'
channels = ['LQE', 'LQN', 'LFE', 'LFN', 'LFZ']
request = [[network, station, '', ch, start, end] for ch in channels]
request_df = pd.DataFrame(request, columns=fdsn.request_columns)
request_df


In [ ]:
h5_path = Path('/tmp/8P_CAS04_electric_demo.h5')
if h5_path.exists():
    h5_path.unlink()

h5_filename = fdsn.make_mth5_from_fdsn_client(request_df, path='/tmp', interact=False)
h5_filename


In [ ]:
m = MTH5()
m.open_mth5(h5_filename)
survey = m.surveys_group.groups_list[0]
station_obj = m.get_station(station, survey=survey)
run_obj = station_obj.get_run('a')

calibrated = {}
for ch_name in ['ex', 'ey', 'hx', 'hy', 'hz']:
    raw = run_obj.get_channel(ch_name).to_channel_ts()
    cal = raw.remove_instrument_response(plot=False)
    calibrated[ch_name] = cal
    print(f'{ch_name}: {raw.channel_metadata.units} -> {cal.channel_metadata.units}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for ax, name, label in [
    (axes[0], 'ex', 'E_x'),
    (axes[0], 'ey', 'E_y'),
    (axes[1], 'hx', 'H_x'),
    (axes[1], 'hy', 'H_y'),
    (axes[1], 'hz', 'H_z'),
]:
    xr = calibrated[name].to_xarray()
    ax.plot(xr.time.values[:400], xr.values[:400], label=label)
    ax.set_ylabel(calibrated[name].channel_metadata.units)
    ax.legend(loc='upper right', fontsize='small')

axes[0].set_title('EarthScope electric and magnetic field data for CAS04, calibrated to physical units')
axes[1].set_xlabel('time [UTC]')
plt.tight_layout()
plt.show()


## Notes

- The FDSN request uses the archived channel names (`LQE`, `LQN`, `LFE`, ...), while the MTH5 object exposes them internally as `ex`, `ey`, `hx`, `hy`, `hz`.
- The calibration step converts counts to physical units using the metadata response in the archive. For electric fields, the instrument response includes the dipole-length correction, so the final units are `milliVolt per kilometer` (mV/km).
- If the channel metadata are incomplete or the response is missing, the archive may not calibrate cleanly; in that case, the right workflow is to flag the gap and avoid silently inventing calibration values.
- This notebook is intentionally a prototype for the `EarthScope electric-field tutorial` milestone and should be expanded with a final narrative section, validation checks, and a documented station/date choice before merging upstream.
